# Phase 2: EEG Signal Quality Analysis and Visualization

This notebook orchestrates the Phase 2 signal-quality pipeline (`imagined_speech.data.signal_quality`) on Google Colab or locally.

### Scientific Safeguards
- **No Preprocessing Applied**: Raw EDF recordings are read in-memory in their original physical units without bandpass/notch filtering, resampling, ICA, baseline correction, or normalization.
- **No Unverified Assumptions**: Filename prefixes are retained strictly as `unverified_candidate_prefix`, and imagined-speech timing intervals are not inferred without verified protocol markers.
- **Measured vs. Heuristic Separation**: Measured quantities (`measured_*`) are reported separately from heuristic diagnostic flags (`heuristic_*`). Heuristic flags indicate channels warranting inspection, not definitive proof of physiological artifacts.

## 1. Mount Google Drive
Because Google Colab runtimes are ephemeral and do not persist local files across sessions, mount Google Drive to access the original dataset archive and store persistent outputs.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Google Colab: {IN_COLAB}')

## 2. Clone or Update the Repository
Ensure the latest version of the `imagined-speech-eeg` repository is available in the current session.

In [ ]:
REPO_URL = 'https://github.com/ACT2039/imagined-speech-eeg.git'
COLAB_REPO_DIR = Path('/content/imagined-speech-eeg')

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        !git clone {REPO_URL} {COLAB_REPO_DIR}
    else:
        !git -C {COLAB_REPO_DIR} pull --ff-only
    os.chdir(COLAB_REPO_DIR)

print('Working directory:', Path.cwd())

## 3. Install the Package
Install `imagined-speech-recognition` and its dependencies in editable mode.

In [ ]:
if IN_COLAB:
    !python -m pip install -q -e ".[dev]"

## 4. Configure Dataset Root and Persistent Output Paths
If running in a fresh Colab session where `/content/data/raw` does not yet exist, extract the read-only dataset archive from Google Drive into local runtime storage.

In [ ]:
# Override via environment variables or edit to match your Google Drive layout
DRIVE_ZIP_PATH = Path(
    os.environ.get(
        'ISR_DRIVE_ZIP',
        '/content/drive/MyDrive/imagined_speech_project/raw/imagined_speech_edf.zip',
    )
)
DATA_ROOT = Path(os.environ.get('ISR_DATA_ROOT', '/content/data/raw' if IN_COLAB else './data/raw'))
PHASE1_AUDIT_DIR = Path(
    os.environ.get(
        'ISR_PHASE1_AUDIT_DIR',
        '/content/drive/MyDrive/imagined_speech_project/outputs/phase1_audit'
        if IN_COLAB
        else './outputs/phase1_audit',
    )
)
PHASE2_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_project/outputs/phase2_signal_quality'
        if IN_COLAB
        else './outputs/phase2_signal_quality',
    )
)
CONFIG_PATH = Path('configs/default_config.yaml')

if IN_COLAB and not DATA_ROOT.exists() and DRIVE_ZIP_PATH.is_file():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_ROOT)
    print(f'Extracted dataset archive {DRIVE_ZIP_PATH} -> {DATA_ROOT}')

print(f'DATA_ROOT:         {DATA_ROOT}')
print(f'PHASE1_AUDIT_DIR:  {PHASE1_AUDIT_DIR}')
print(f'PHASE2_OUTPUT_DIR: {PHASE2_OUTPUT_DIR}')

## 5. Load and Validate the Existing Audit Manifest
Load `manifest.csv` produced in Phase 1 (or generate it automatically if not yet present in `PHASE1_AUDIT_DIR`).

In [ ]:
from imagined_speech.config import load_config
from imagined_speech.data.edf_audit import run_dataset_audit
from imagined_speech.data.signal_quality import (
    load_and_validate_manifest,
    run_signal_quality_analysis,
    select_representative_recordings,
)
from imagined_speech.utils import set_global_seed, setup_logging

config = load_config(
    config_path=CONFIG_PATH if CONFIG_PATH.exists() else None,
    data_root=DATA_ROOT,
    output_dir=PHASE2_OUTPUT_DIR,
    recordings_per_task=3,
)
setup_logging(config.log_level)
set_global_seed(config.seed)

manifest_path = PHASE1_AUDIT_DIR / 'manifest.csv'
if not manifest_path.is_file():
    print(f'Phase 1 manifest not found at {manifest_path}; running Phase 1 dataset audit...')
    phase1_cfg = load_config(
        config_path=CONFIG_PATH if CONFIG_PATH.exists() else None,
        data_root=DATA_ROOT,
        output_dir=PHASE1_AUDIT_DIR,
    )
    run_dataset_audit(phase1_cfg, save_outputs=True)

manifest_df = load_and_validate_manifest(manifest_path)
print(f'Validated manifest rows: {len(manifest_df)}')
display(manifest_df.head())

## 6. Deterministically Select Representative Recordings
Select representative recordings across `Char`, `Digit`, and `Image`, spanning diverse labels and durations.

In [ ]:
import pandas as pd
from dataclasses import asdict

selected_inputs = select_representative_recordings(
    manifest_df=manifest_df,
    data_root=config.data_root,
    expected_tasks=config.discovery.expected_tasks,
    recordings_per_task=config.signal_quality.recordings_per_task,
    specific_recordings=config.signal_quality.specific_recordings,
    seed=config.seed,
)
display(pd.DataFrame([asdict(s) for s in selected_inputs]))

## 7. Run the EEG Signal Quality Analysis
Inspect EDF headers and raw signals in original physical units, compute per-channel statistics and heuristic flags, and generate figures and reports.

In [ ]:
import json

sq_result = run_signal_quality_analysis(
    config=config,
    manifest_source=manifest_df,
    output_dir=PHASE2_OUTPUT_DIR,
    save_outputs=True,
)
print(json.dumps(sq_result.summary, indent=2))

## 8. Display Summary Tables and Generated Plots

In [ ]:
from IPython.display import Image, Markdown, display

stats_df = pd.read_csv(PHASE2_OUTPUT_DIR / 'signal_statistics.csv')
display(stats_df.head(20))

display(Markdown((PHASE2_OUTPUT_DIR / 'signal_quality_report.md').read_text(encoding='utf-8')))

for fig_path in sq_result.generated_figures:
    print(f'Figure: {fig_path}')
    display(Image(filename=fig_path))

## 9. Verify Persistent Storage in Google Drive
Confirm that `signal_statistics.csv`, `signal_quality_summary.json`, `signal_quality_report.md`, and `figures/` are saved to the persistent output directory.

In [ ]:
saved_files = sorted(p.relative_to(PHASE2_OUTPUT_DIR).as_posix() for p in PHASE2_OUTPUT_DIR.rglob('*') if p.is_file())
print(f'Total files saved to {PHASE2_OUTPUT_DIR}: {len(saved_files)}')
for rel_f in saved_files:
    print(' -', rel_f)